# A11 — Pipeline End-to-End: Modelo Final

**Objetivo:** Consolidar todo o projeto em um pipeline integrado e reprodutível,
conectando preparação de dados, treinamento, avaliação, inferência e geração
automática de resultados.

## Estrutura do pipeline

| Etapa | Módulo | Descrição |
|-------|--------|-----------|
| 1. Setup | `config.yaml` + `reprodutibilidade` | Seed global, caminhos, hiperparâmetros |
| 2. Pré-processamento | `src.preprocessing` | CSV → tensor 4D, split por image_id, tf.data |
| 3. Treinamento | `src.training` | MobileNetV2 em 2 fases (head + fine-tuning) |
| 4. Avaliação | `src.evaluation` | Métricas, curvas ROC/PR, matrizes de confusão |
| 5. Inferência | `src.inference` | Predição em lote, exportação padronizada |

**Execução simples:** basta executar todas as células em sequência (`Run All`).

## 1. Setup — Reprodutibilidade e configuração

In [ ]:
"""
Célula 1 — Configuração do ambiente e seed global.

Garante que todos os experimentos sejam reprodutíveis fixando seeds
em Python, NumPy e TensorFlow, além de ativar determinismo de operações.
"""
import os
import sys
import json
import logging
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Caminhos do projeto ────────────────────────────────────────────────────
# Notebook está em: artefatos/a11_modelo_final/notebooks/
# Raiz do projeto:  ../../..  (3 níveis acima)
NOTEBOOK_DIR = Path.cwd() if "__file__" not in dir() else Path(__file__).parent
ARTIFACT_ROOT = Path(os.path.abspath(os.path.join(NOTEBOOK_DIR, "..")))
PROJECT_ROOT = Path(os.path.abspath(os.path.join(ARTIFACT_ROOT, "..", "..")))

# Adicionar raiz do projeto ao sys.path para imports de src/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Logging ────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("a11_pipeline")

# ── Reprodutibilidade ─────────────────────────────────────────────────────
from src.reprodutibilidade import set_global_seed

SEED = 42
set_global_seed(SEED)

# ── Carregar configuração ──────────────────────────────────────────────────
from artefatos.a11_modelo_final.src.preprocessing import load_config

CONFIG_PATH = ARTIFACT_ROOT / "config.yaml"
cfg = load_config(CONFIG_PATH)

logger.info("Raiz do projeto: %s", PROJECT_ROOT)
logger.info("Raiz do artefato: %s", ARTIFACT_ROOT)
logger.info("Seed: %d", cfg["seed"])
logger.info("Config carregada: %s", CONFIG_PATH)

In [ ]:
# ── Verificação de caminhos de dados ───────────────────────────────────────
DATASET_CSV = PROJECT_ROOT / "data" / "pixels_dataset.csv"
EXTRACTED_CODES_JSON = PROJECT_ROOT / "data" / "extracted_codes.json"

assert DATASET_CSV.exists(), f"Dataset não encontrado: {DATASET_CSV}"
assert EXTRACTED_CODES_JSON.exists(), f"Códigos não encontrados: {EXTRACTED_CODES_JSON}"

# Diretórios de saída (criados automaticamente)
OUTPUT_MODELS = ARTIFACT_ROOT / cfg["paths"]["outputs_models"]
OUTPUT_METRICS = ARTIFACT_ROOT / cfg["paths"]["outputs_metrics"]
OUTPUT_VIZ = ARTIFACT_ROOT / cfg["paths"]["outputs_viz"]
OUTPUT_PREDS = ARTIFACT_ROOT / cfg["paths"]["outputs_preds"]

for d in (OUTPUT_MODELS, OUTPUT_METRICS, OUTPUT_VIZ, OUTPUT_PREDS):
    d.mkdir(parents=True, exist_ok=True)

logger.info("Dataset: %s", DATASET_CSV)
logger.info("Códigos: %s", EXTRACTED_CODES_JSON)
logger.info("Saídas → modelos: %s", OUTPUT_MODELS)
logger.info("Saídas → métricas: %s", OUTPUT_METRICS)
logger.info("Saídas → visualizações: %s", OUTPUT_VIZ)
logger.info("Saídas → predições: %s", OUTPUT_PREDS)
print("\n✔ Setup concluído com sucesso.")

## 2. Pré-processamento — Dados → Tensores → tf.data

Etapas automatizadas:
1. Carrega `pixels_dataset.csv` e extrai labels de `extracted_codes.json`.
2. Converte colunas `pixel_*` em tensor 4D `(N, H, W, C)`.
3. Split estratificado por `image_id` (treino / validação / teste) para evitar vazamento de dados.
4. Normalização z-score por canal (ajustada apenas no treino).
5. Data augmentation no treino (flip, rotação, contraste).

In [ ]:
from artefatos.a11_modelo_final.src.preprocessing import run_preprocessing

prep_result = run_preprocessing(
    cfg,
    dataset_csv=DATASET_CSV,
    extracted_codes_json=EXTRACTED_CODES_JSON,
)

splits = prep_result["splits"]
tf_data = prep_result["tf_data"]

# Resumo do pré-processamento
print("\n" + "═" * 60)
print("RESUMO DO PRÉ-PROCESSAMENTO")
print("═" * 60)
print(f"  Amostras treino:     {splits['X_train'].shape[0]:>6d}  |  shape: {splits['X_train'].shape}")
print(f"  Amostras validação:  {splits['X_val'].shape[0]:>6d}  |  shape: {splits['X_val'].shape}")
print(f"  Amostras teste:      {splits['X_test'].shape[0]:>6d}  |  shape: {splits['X_test'].shape}")
print(f"  Input shape (CNN):   {tf_data['train_meta']['input_shape']}")
print(f"  Normalização:        {tf_data['train_meta']['normalization']}")
print(f"  Augmentação (treino):{tf_data['train_meta']['augment']}")
print("═" * 60)
print("\n✔ Pré-processamento concluído.")